# AI Circuit — Kaggle Setup

Supports both workflows:
- **Human+Agent**: you prepare data via `utils/data_prep.py`, agent handles working/training
- **Full Agent**: fully autonomous — LLM picks classes, preps data, trains

### Prerequisites
- Add the **H&M dataset** to this notebook: *Add Data → H&M Personalized Fashion Recommendations*
- Add your **OpenAI API key** as a Kaggle Secret named `OPENAI_API_KEY`
- Enable **GPU** accelerator in notebook settings (T4 recommended)

In [11]:
# ── 1. Clone repo and install dependencies ────────────────────────────────
import subprocess, sys

REPO_URL = "https://github.com/saichandra1199/ai-circuit.git"
REPO_DIR = "/kaggle/working/training/ai_circuit"

subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
%cd /kaggle/working/training/ai_circuit

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("Setup complete.")

Cloning into '/kaggle/working/training/ai_circuit'...


/kaggle/working/training/ai_circuit
Setup complete.


In [12]:
# ── 2. Set API key from Kaggle Secrets ────────────────────────────────────
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["OPENAI_API_KEY"] = secrets.get_secret("OPENAI_API_KEY")
print("API key loaded.")

API key loaded.


In [13]:
# ── 3. Verify GPU ─────────────────────────────────────────────────────────
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


In [14]:
! cd /kaggle/working/training/ai_circuit/ && ls -la

total 100
drwxr-xr-x 6 root root  4096 Jul 13 08:43 .
drwxr-xr-x 3 root root  4096 Jul 13 08:43 ..
drwxr-xr-x 2 root root  4096 Jul 13 08:43 agents
drwxr-xr-x 2 root root  4096 Jul 13 08:43 docs
drwxr-xr-x 8 root root  4096 Jul 13 08:43 .git
-rw-r--r-- 1 root root   194 Jul 13 08:43 .gitignore
-rw-r--r-- 1 root root  4681 Jul 13 08:43 kaggle_guide.md
-rw-r--r-- 1 root root  7970 Jul 13 08:43 kaggle_setup.ipynb
-rw-r--r-- 1 root root   440 Jul 13 08:43 pyproject.toml
-rw-r--r-- 1 root root     5 Jul 13 08:43 .python-version
-rw-r--r-- 1 root root  6655 Jul 13 08:43 README.md
-rw-r--r-- 1 root root   199 Jul 13 08:43 requirements.txt
-rw-r--r-- 1 root root  2134 Jul 13 08:43 run_full_agent.py
-rw-r--r-- 1 root root   471 Jul 13 08:43 run_human_agent.py
-rw-r--r-- 1 root root  5978 Jul 13 08:43 training_config.yaml
-rw-r--r-- 1 root root 19121 Jul 13 08:43 train.py
drwxr-xr-x 2 root root  4096 Jul 13 08:43 utils


---
## Workflow A — Human + Agent
You prepare the dataset, agent handles working/training loop.

In [16]:
# ── A1. Prepare dataset ───────────────────────────────────────────────────
# Edit max_per_class to control dataset size (500 = fast test, None = full)
import sys
sys.path.insert(0, "/kaggle/working/training/ai_circuit")

from utils.data_prep import prepare

prepare(
    raw_data_dir="/kaggle/input/competitions/h-and-m-personalized-fashion-recommendations",
    output_dir="/kaggle/working/training/ai_circuit/data/prepared",
    max_per_class=500,   # increase for full run
)

Rows in 5 classes: 92286
product_group_name
Garment Upper body    42741
Garment Lower body    19812
Garment Full body     13292
Accessories           11158
Shoes                  5283

Images on disk: 91887
Validating images...


Corrupt dropped: 0  →  91887 remain
Deduplicating...


/kaggle/training/ai_circuit/utils/data_prep.py:134: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(min(len(g), max_per_class), random_state=seed))


Duplicates removed: 6100  →  85787 remain

After cap (500/class):
product_group_name
Accessories           500
Garment Full body     500
Garment Lower body    500
Garment Upper body    500
Shoes                 500

Copying images...


Computing dataset mean/std...



────────────────────────────────────────────────────────────
Saved → /kaggle/working/training/ai_circuit/data/prepared
  train:  2000  |  Accessories=400  body=400  body=400  body=400  Shoes=400
  val  :   250  |  Accessories=50  body=50  body=50  body=50  Shoes=50
  test :   250  |  Accessories=50  body=50  body=50  body=50  Shoes=50
Weights: {'Accessories': 1.0, 'body': 1.0, 'Shoes': 1.0}
Mean: [0.7763, 0.7597, 0.7566]
Std:  [0.1943, 0.2105, 0.21]
────────────────────────────────────────────────────────────


{'data_dir': '/kaggle/working/training/ai_circuit/data/prepared',
 'class_mapping': '/kaggle/working/training/ai_circuit/data/prepared/class_mapping.json',
 'class_weights': '/kaggle/working/training/ai_circuit/data/prepared/class_weights.json',
 'dataset_stats': '/kaggle/working/training/ai_circuit/data/prepared/dataset_stats.json'}

In [25]:
# ── A2. Patch config for Kaggle and run agent ─────────────────────────────
import yaml

with open("training_config.yaml") as f:
    cfg = yaml.safe_load(f)

# Kaggle-specific overrides
cfg["paths"]["data_dir"]        = "/kaggle/working/training/ai_circuit/data/prepared"
cfg["paths"]["class_weights"]   = None
cfg["paths"]["class_mapping"]   = None
cfg["training"]["num_workers"]  = 4
cfg["agent"]["experiment_name"] = "kaggle_human_agent"   # edit as needed
cfg["agent"]["max_iterations"]  = 3                      # edit as needed

from agents.training_agent import run as run_training
run_training(config_path=cfg, max_iterations=cfg["agent"]["max_iterations"],
             target_f1=cfg["agent"]["target_f1"], workflow="human+agent")

Session dir: experiments/20260713_095346_kaggle_human_agent
LLM model:   gpt-4o-mini
Workflow:    human+agent

Run 1 | config → experiments/20260713_095346_kaggle_human_agent/run_1/config.yaml

[agent_run_1] convnext_base.fb_in22k_ft_in1k  train=75  val=75  epochs=1

Ep   1/1 | loss=1.9492 acc=0.0933 | val_loss=1.7272 val_f1=0.2462 | lr=1.00e-06 | 35s | ETA 0s
  ★  New best val F1: 0.2462

Test | acc=0.3867  macro_f1=0.2267

Saved to: experiments/20260713_095346_kaggle_human_agent/run_1

Training completed in 90s

[Notes saved → experiments/20260713_095346_kaggle_human_agent/run_1/notes.md]

[Improve] Proposed changes: {
  "training.epochs": 10,
  "loss.use_class_weights": true,
  "loss.label_smoothing": 0.1,
  "augmentations.random_erasing": true,
  "augmentations.random_erasing_prob": 0.3
}

Run 2 | config → experiments/20260713_095346_kaggle_human_agent/run_2/config.yaml
Changes: {
  "training.epochs": 10,
  "loss.use_class_weights": true,
  "loss.label_smoothing": 0.1,
  "augmentat

{'session_dir': 'experiments/20260713_095346_kaggle_human_agent',
 'llm_model': 'gpt-4o-mini',
 'workflow': 'human+agent',
 'run_num': 3,
 'base_config': {'agent': {'experiment_name': 'kaggle_human_agent',
   'max_iterations': 3,
   'target_f1': 0.9,
   'llm_model': 'gpt-4o-mini'},
  'model': {'backbone': 'convnext_base.fb_in22k_ft_in1k',
   'pretrained': True,
   'checkpoint': None,
   'image_size': 224,
   'dropout': 0.3},
  'training': {'epochs': 1,
   'batch_size': 32,
   'num_workers': 4,
   'mixed_precision': True,
   'early_stopping_patience': 5,
   'early_stopping_min_delta': 0.001,
   'max_samples_per_class': 15},
  'optimizer': {'type': 'adamw',
   'lr': 0.0003,
   'weight_decay': 0.01,
   'momentum': 0.9},
  'scheduler': {'type': 'cosine',
   'min_lr': 1e-06,
   'step_size': 7,
   'gamma': 0.1,
   'warmup_epochs': 0},
  'loss': {'type': 'weighted_ce',
   'label_smoothing': 0.0,
   'focal_gamma': 2.0,
   'use_class_weights': True},
  'sampler': {'use_weighted': True},
  'augm

---
## Workflow B — Full Agent
Fully autonomous — LLM selects classes, prepares data, trains.

In [32]:
# ── B1. Patch config and run full agent ───────────────────────────────────
import yaml, sys
sys.path.insert(0, "/kaggle/working/training/ai_circuit")

with open("training_config.yaml") as f:
    cfg = yaml.safe_load(f)

# Kaggle-specific overrides
cfg["data_prep"]["raw_data_dir"]       = "/kaggle/input/competitions/h-and-m-personalized-fashion-recommendations"
cfg["data_prep"]["max_train_per_class"] = 500   # increase for full run
cfg["training"]["num_workers"]          = 4
cfg["agent"]["experiment_name"]         = "kaggle_full_agent"   # edit as needed
cfg["agent"]["max_iterations"]          = 3

from agents.data_prep_agent import prepare_data
from agents.training_agent import run as run_training

exp_name = cfg["agent"].get("experiment_name") or "auto"
data_prep_cfg = cfg["data_prep"]

data_paths = prepare_data(
    raw_data_dir=data_prep_cfg["raw_data_dir"],
    output_dir=f"/kaggle/working/training/ai_circuit/data/{exp_name.replace(' ', '_')}",
    max_train_per_class=data_prep_cfg.get("max_train_per_class"),
    llm_model=cfg["agent"]["llm_model"],
    instructions=data_prep_cfg.get("instructions"),
    force_classes=data_prep_cfg.get("force_classes"),
)

cfg["paths"]["data_dir"]      = data_paths["data_dir"]
cfg["paths"]["class_weights"] = None
cfg["paths"]["class_mapping"] = None

run_training(config_path=cfg, max_iterations=cfg["agent"]["max_iterations"],
             target_f1=cfg["agent"]["target_f1"], workflow="full_agent")

Label distribution:
  Garment Upper body: 42741
  Garment Lower body: 19812
  Garment Full body: 13292
  Accessories: 11158
  Underwear: 5490
  Shoes: 5283
  Swimwear: 3127
  Socks & Tights: 2442
  Nightwear: 1899
  Unknown: 121
  Underwear/nightwear: 54
  Cosmetic: 49
  Bags: 25
  Items: 17
  Furniture: 13
  Garment and Shoe care: 9
  Stationery: 5
  Interior textile: 3
  Fun: 2

Using forced classes: ['Garment Upper body', 'Garment Lower body', 'Garment Full body', 'Accessories', 'Shoes']

91887 items with images found (399 missing images).
Copying images...


/kaggle/working/training/ai_circuit/agents/data_prep_agent.py:69: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.head(n))
/kaggle/working/training/ai_circuit/agents/data_prep_agent.py:69: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.head(n))
/kaggle/working/training/ai_circuit/agents/data_prep_agent.py:69: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. Th


────────────────────────────────────────────────────────────
Data prepared → /kaggle/working/training/ai_circuit/data/kaggle_full_agent
  train:   2500 imgs  |  Accessories=500  Garment Full body=500  Garment Lower body=500  Garment Upper body=500  Shoes=500
  val  :    310 imgs  |  Accessories=62  Garment Full body=62  Garment Lower body=62  Garment Upper body=62  Shoes=62
  test :    310 imgs  |  Accessories=62  Garment Full body=62  Garment Lower body=62  Garment Upper body=62  Shoes=62
Class weights: {'Accessories': 1.0, 'Garment Full body': 1.0, 'Garment Lower body': 1.0, 'Garment Upper body': 1.0, 'Shoes': 1.0}
────────────────────────────────────────────────────────────

Generating data prep notes...
Data prep notes → /kaggle/working/training/ai_circuit/data/kaggle_full_agent/data_prep_notes.md

Session dir: experiments/20260713_101932_kaggle_full_agent
LLM model:   gpt-4o-mini
Workflow:    full_agent

Run 1 | config → experiments/20260713_101932_kaggle_full_agent/run_1/config.

{'session_dir': 'experiments/20260713_101932_kaggle_full_agent',
 'llm_model': 'gpt-4o-mini',
 'workflow': 'full_agent',
 'run_num': 3,
 'base_config': {'agent': {'experiment_name': 'kaggle_full_agent',
   'max_iterations': 3,
   'target_f1': 0.9,
   'llm_model': 'gpt-4o-mini'},
  'model': {'backbone': 'convnext_base.fb_in22k_ft_in1k',
   'pretrained': True,
   'checkpoint': None,
   'image_size': 224,
   'dropout': 0.3},
  'training': {'epochs': 1,
   'batch_size': 32,
   'num_workers': 4,
   'mixed_precision': True,
   'early_stopping_patience': 5,
   'early_stopping_min_delta': 0.001,
   'max_samples_per_class': 15},
  'optimizer': {'type': 'adamw',
   'lr': 0.0003,
   'weight_decay': 0.01,
   'momentum': 0.9},
  'scheduler': {'type': 'cosine',
   'min_lr': 1e-06,
   'step_size': 7,
   'gamma': 0.1,
   'warmup_epochs': 0},
  'loss': {'type': 'weighted_ce',
   'label_smoothing': 0.0,
   'focal_gamma': 2.0,
   'use_class_weights': True},
  'sampler': {'use_weighted': True},
  'augment

---
## Results
Experiment logs and checkpoints saved under `/kaggle/working/training/ai_circuit/experiments/`

In [37]:
# ── View experiment log ───────────────────────────────────────────────────
import json, glob

logs = sorted(glob.glob("/kaggle/working/training/ai_circuit/experiments/*/experiment_log.json"))
for log_path in logs:
    print(f"\n=== {log_path} ===")
    with open(log_path) as f:
        for run in json.load(f):
            print(f"  Run {run['run']} | backbone={run['backbone']} | macro_f1={run['macro_f1']:.4f} | epochs={run['epochs_trained']}")


=== /kaggle/working/training/ai_circuit/experiments/20260713_095346_kaggle_human_agent/experiment_log.json ===
  Run 1 | backbone=convnext_base.fb_in22k_ft_in1k | macro_f1=0.2267 | epochs=1
  Run 2 | backbone=convnext_base.fb_in22k_ft_in1k | macro_f1=0.6931 | epochs=10
  Run 3 | backbone=convnext_base.fb_in22k_ft_in1k | macro_f1=0.7738 | epochs=11

=== /kaggle/working/training/ai_circuit/experiments/20260713_101932_kaggle_full_agent/experiment_log.json ===
  Run 1 | backbone=convnext_base.fb_in22k_ft_in1k | macro_f1=0.4550 | epochs=1
  Run 2 | backbone=convnext_base.fb_in22k_ft_in1k | macro_f1=0.6873 | epochs=5
  Run 3 | backbone=convnext_base.fb_in22k_ft_in1k | macro_f1=0.9092 | epochs=10


## Compare Experiments

In [35]:
!cd /kaggle/working/training/ai_circuit/experiments/ && ls -la

total 24
drwxr-xr-x 4 root root 4096 Jul 13 10:19 .
drwxr-xr-x 8 root root 4096 Jul 13 09:53 ..
drwxr-xr-x 5 root root 4096 Jul 13 09:57 20260713_095346_kaggle_human_agent
drwxr-xr-x 5 root root 4096 Jul 13 10:21 20260713_101932_kaggle_full_agent
-rw-r--r-- 1 root root 5184 Jul 13 10:23 master_log.json


In [40]:
# ── Compare two experiment sessions ──────────────────────────────────────────
import sys
sys.path.insert(0, "/kaggle/working/training/ai_circuit")

from utils.compare_experiments import load_session, compare

# Set to the session dirs you want to compare (under experiments/)
EXPERIMENTS_BASE = "/kaggle/working/training/ai_circuit/experiments"

workflow_a = "20260713_095346_kaggle_human_agent"   # edit
workflow_b = "20260713_101932_kaggle_full_agent"    # edit

import os
from pathlib import Path

def find_session(workflow_name: str, base: str) -> str:
  """Find latest session dir matching workflow name in master_log."""
  import json
  master = Path(base) / "master_log.json"
  if master.exists():
      entries = json.loads(master.read_text())
      matches = [e for e in entries if e.get("workflow") == workflow_name or
                 workflow_name in e.get("experiment_name", "")]
      if matches:
          return str(Path(base) / Path(matches[-1]["session_dir"]).name)
  # fallback: glob for dir containing the name
  hits = sorted(Path(base).glob(f"*{workflow_name}*"))
  if hits:
      return str(hits[-1])
  raise FileNotFoundError(f"No session found for workflow: {workflow_name}")

sessions = []
for wf in [workflow_a, workflow_b]:
  try:
      path = find_session(wf, EXPERIMENTS_BASE)
      sessions.append(load_session(path))
  except FileNotFoundError as e:
      print(f"Warning: {e}")

compare(sessions)


══════════════════════════════════════════════════════════════════════
  COMPARISON SUMMARY
══════════════════════════════════════════════════════════════════════
  Session                        Workflow         Runs  Best F1 Best backbone
  ------------------------------ ---------------- ---- -------- ------------------------------
  20260713_095346_kaggle_human_agent [Human+Agent]       3   0.7738    convnext_base.fb_in22k_ft_in1k
  20260713_101932_kaggle_full_agent [Full Agent]        3   0.9092 ★  convnext_base.fb_in22k_ft_in1k

  Winner: 20260713_101932_kaggle_full_agent (macro F1 = 0.9092)
  Best checkpoint: experiments/20260713_101932_kaggle_full_agent/run_3/best_model.pth

════════════════════════════════════════════════════════════
  20260713_095346_kaggle_human_agent  [Human+Agent]
════════════════════════════════════════════════════════════
  Runs: 3   Best macro F1: 0.7738

  run  1    test F1=0.2267 █████░░░░░░░░░░░░░░░  val F1=0.2462  ep= 1
         backbone: convnext_b